# Preprocessing and Feature Engineering
---
## Tasks done in this notebook:
- Feature Creation, Scaling and Encoding
- Class Imbalance Handling 
- Train/Val/Test sets preparation

**Note: All statistics (mean, percentiles, encoders) must be fit on Train set only in order to prevent leakage**

*This notebook working is dependent on the EDA findings*

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import logging

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

RANDOM_STATE = 42 

### Initialization:
- Data Loading 
- Integrity Check 
- Initial Inspection of Data

In [3]:
credit_data_df = pd.read_csv('../data/interim/cleaned/credit_card_fraud_10k_cleaned.csv')
credit_data_df.columns = credit_data_df.columns.str.strip()
print("Dataset Shape:", credit_data_df.shape)
print("\nDataset Info:")
print(credit_data_df.info())
print("\nDataset Description:")
print(credit_data_df.describe())

print("\nDuplicate Rows:", credit_data_df.duplicated().sum())
print("\nMissing Values in Each Column:")
print(credit_data_df.isnull().sum())

continuous_cols = ['amount', 'transaction_hour', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']
binary_cols = ['foreign_transaction', 'location_mismatch']
categorical_cols = ['merchant_category']
target_col = ['is_fraud']

target_distribution = credit_data_df[target_col].value_counts(normalize=True)
print("\nTarget Variable Distribution:")
print(target_distribution)

Dataset Shape: (10000, 10)

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       10000 non-null  int64  
 1   amount               10000 non-null  float64
 2   transaction_hour     10000 non-null  int64  
 3   merchant_category    10000 non-null  str    
 4   foreign_transaction  10000 non-null  int64  
 5   location_mismatch    10000 non-null  int64  
 6   device_trust_score   10000 non-null  float64
 7   velocity_last_24h    10000 non-null  float64
 8   cardholder_age       10000 non-null  int64  
 9   is_fraud             10000 non-null  int64  
dtypes: float64(3), int64(6), str(1)
memory usage: 781.4 KB
None

Dataset Description:
       transaction_id        amount  transaction_hour  foreign_transaction  \
count     10000.00000  10000.000000      10000.000000         10000.000000   
mean       5000.50

### Stratified Train/Val/Test Split (Pre-Engineering)

- Splitting the dataset to train with 70% of data, val with 15% of data and test with 15% of the data
- Why split first? Prevents leakage when computing percentiles, merchant baselines, or encoders

In [4]:
train_size = 0.7 
val_size = 0.15
test_size = 0.15

X = credit_data_df.drop(columns=target_col)
Y = credit_data_df[target_col]

X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, 
    Y, 
    test_size=(1 - train_size), 
    random_state=RANDOM_STATE, 
    stratify=Y
)   

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=(test_size / (test_size + val_size)),
    random_state=RANDOM_STATE,
    stratify=Y_temp
)

print("\nTraining Set Shape:", X_train.shape)
print("Validation Set Shape:", X_val.shape)
print("Test Set Shape:", X_test.shape)

split_data = {
    'X_train': X_train,
    'Y_train': Y_train,
    'X_val': X_val,
    'Y_val': Y_val,
    'X_test': X_test,
    'Y_test': Y_test
}   

for name, data in split_data.items():
    print(f"{name} - Shape: {data.shape}, Class Distribution:\n{data.value_counts(normalize=True)}\n")

assert X_train.shape[0] == Y_train.shape[0], "Mismatch in training set sizes"
assert X_val.shape[0] == Y_val.shape[0], "Mismatch in validation set sizes"
assert X_test.shape[0] == Y_test.shape[0], "Mismatch in test set sizes"
assert len(X_train) + len(X_val) + len(X_test) == len(credit_data_df), "Total samples in splits do not match original dataset"
assert len(Y_train) + len(Y_val) + len(Y_test) == len(credit_data_df), "Total samples in target splits do not match original dataset"


Training Set Shape: (6999, 9)
Validation Set Shape: (1500, 9)
Test Set Shape: (1501, 9)
X_train - Shape: (6999, 9), Class Distribution:
transaction_id  amount  transaction_hour  merchant_category  foreign_transaction  location_mismatch  device_trust_score  velocity_last_24h  cardholder_age
4344            315.12  17                Food               0                    0                  78.0                1.0                46                0.000143
2144            67.93   6                 Electronics        0                    0                  45.0                1.0                67                0.000143
9845            22.07   13                Clothing           0                    0                  33.0                4.0                44                0.000143
2349            162.10  6                 Food               0                    0                  78.0                4.0                38                0.000143
2091            268.99  1               

### Handling Class Imbalance using SMOTE 

In [5]:
X_train_smote_input = X_train.copy()
X_train_smote_input = pd.get_dummies(X_train_smote_input, columns=['merchant_category'], drop_first=True)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, Y_train_smote = smote.fit_resample(X_train_smote_input, Y_train)

print("Before SMOTE:")
print(Y_train.value_counts())

print("\nAfter SMOTE:")
print(Y_train_smote.value_counts())

df_smote_exported = pd.DataFrame(X_train_smote, columns=X_train_smote_input.columns)
df_smote_exported['is_fraud'] = Y_train_smote
df_smote_exported.to_csv('../data/interim/smote/X_train_smote_eda.csv', index=False)

Before SMOTE:
is_fraud
0           6893
1            106
Name: count, dtype: int64

After SMOTE:
is_fraud
0           6893
1           6893
Name: count, dtype: int64


## Feature Engineering (Train-Derived Statistics)
---
### X_train is the only set that will be used while doing the feature engineering

Engineered features and their justifications:
- `log_amount`: handles right skew of amount.
- `is_night_transaction`: 12 AM to 5 AM found to be the burst window. (boolean flag)
- `velocity_per_hour`: identify sudden spikes in activity relative to the time of day.
- `is_high_velocity_low_trust`: fraud found to be more often at high velocity and low trust. (boolean flag)
- `high_risk_abroad`: fraud is significantly higher in foreign markets regardless of price.
- `relative_amount`: show if a purchase is "out of character" for that specific category.


How to calculate?
- `log_amount`: log transformation for the amount.
- `is_night_transaction`: if between 12 AM to 5 AM set with 1, else set with 0.
- `velocity_per_hour`: `velocity_last_24h`/(`transaction_hour` + 1).
- `high_risk_abroad`: `foreign_transaction` * `location_mismatch`.
- `is_high_velocity_low_trust`: if (velocity_last_24h > threshold) & (device_trust_score < threshold) then set with 1, else set with 0. Threshold will be a percentile from the velocity so that it becomes self adjusting.
- `relative_amount`: `amount` / `average_merchant_category_amount`. 
- `average_merchant_category_amount`: mean transaction amount for each specific merchant category.


Note: Encode the merchant category using an one hot encoder in order to use it later with models


In [6]:
train_avg_merchant_amount = X_train.groupby('merchant_category')['amount'].mean()
global_avg_merchant_amount = train_avg_merchant_amount.mean()

train_high_velocity_threshold = X_train['velocity_last_24h'].quantile(0.9)
train_low_velocity_threshold = X_train['velocity_last_24h'].quantile(0.1)


def feature_engineering(df):
    df = df.copy()

    df['log_amount'] = np.log1p(df['amount'])
    df['is_night_transaction'] = df['transaction_hour'].between(0, 5).astype(int)
    df['velocity_per_hour'] = df['velocity_last_24h'] / np.maximum(df['transaction_hour'], 1)

    df['high_risk_abroad'] = ((df['foreign_transaction'] == 1) &
                             (df['location_mismatch'] == 1)).astype(int)

    df['is_high_velocity_low_trust'] = (
        (df['velocity_last_24h'] > train_high_velocity_threshold) &
        (df['device_trust_score'] < train_low_velocity_threshold)
    ).astype(int)

    df['relevant_amount'] = df['amount'] / (
        df['merchant_category'].map(train_avg_merchant_amount)
        .fillna(global_avg_merchant_amount) + 1e-6
    )

    return df

X_train = feature_engineering(X_train)
X_val   = feature_engineering(X_val)
X_test  = feature_engineering(X_test)

X_train = pd.get_dummies(X_train, columns=['merchant_category'], drop_first=True, dtype=int)
X_val   = pd.get_dummies(X_val, columns=['merchant_category'], drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test, columns=['merchant_category'], drop_first=True, dtype=int)

all_cols = sorted(set(X_train.columns) | set(X_val.columns) | set(X_test.columns))

X_train = X_train.reindex(columns=all_cols, fill_value=0)
X_val   = X_val.reindex(columns=all_cols, fill_value=0)
X_test  = X_test.reindex(columns=all_cols, fill_value=0)

for df in [X_train, X_val, X_test]:
    df.drop(columns=['amount', 'transaction_id', 'merchant_category'],
            inplace=True, errors='ignore')

print("\nX_train columns:", X_train.columns)
print(X_train.describe())


X_train columns: Index(['cardholder_age', 'device_trust_score', 'foreign_transaction',
       'high_risk_abroad', 'is_high_velocity_low_trust',
       'is_night_transaction', 'location_mismatch', 'log_amount',
       'merchant_category_ Electronics      ',
       'merchant_category_ Food             ',
       'merchant_category_ Grocery          ',
       'merchant_category_ Travel           ', 'relevant_amount',
       'transaction_hour', 'velocity_last_24h', 'velocity_per_hour'],
      dtype='str')
       cardholder_age  device_trust_score  foreign_transaction  \
count     6999.000000         6999.000000          6999.000000   
mean        43.378911           61.952565             0.099300   
std         14.935362           21.545389             0.299086   
min         18.000000           25.000000             0.000000   
25%         30.000000           43.500000             0.000000   
50%         43.000000           62.000000             0.000000   
75%         56.000000          

## Now the X_train , X_val and X_test contain
---

| Column | Data Type | Description | Example Values|
|--------|-----------|-------------|---------------|
| transaction_hour | Integer | Hour of day when transaction occurred (24-hour format) | 0 - 23 |
| foreign_transaction | Binary | Whether transaction occurred in a foreign country | 0 = Domestic, 1 = Foreign |
| location_mismatch | Binary | Whether transaction location differs from cardholder's typical location | 0 = Match, 1 = Mismatch |
| device_trust_score | Integer | Trust score of the device used for transaction (higher = more trusted) | 25 - 99 |
| velocity_last_24h | Integer | Number of transactions by this cardholder in past 24 hours | 0 - 9+ |
| cardholder_age | Integer | Age of the cardholder in years | 0 - 69 |
| log_amount | Float | Log scale for the amount as amount had wide range. Calculated as log(amount + 1) | 4.233091, 5.929110 |
| is_night_transaction | Binary | Whether a transaction is done at night between 12 AM and 5 AM | 0 = Normal Hours, 1 = At Night |
| velocity_per_hour | Float | The rate of transactions per hour | 0.058824, 4.000000 |
| high_risk_abroad | Binary | Whether the transaction happened in foreign place and at the same time there is a location mismatch | 0 = Location match or Not in Foreign place or both, 1 = Location mismatch and in a Froreign place |
| is_high_velocity_low_trust | Binary | Whether the transaction happened at high velocity from a low trust device or not | 0 = High trust device or Low velocity or both, 1 = Low trust device and High velocity |
| relevant_amount | Integer | This is the amount relative to specific merchant category | 1.789576, 2.832121 |
| merchant_category_Electronics | Binary | This is due to the one hot encoding. | 0 = Not Electronics , 1 = Electronics |
| merchant_category_Food | Binary | This is due to the one hot encoding. | 0 = Not Food , 1 = Food |
| merchant_category_Grocery | Binary | This is due to the one hot encoding. | 0 = Not Grocery , 1 = Grocery |
| merchant_category_Travel | Binary | This is due to the one hot encoding. | 0 = Not Travel , 1 = Travel |

**Note: since one hote encoding only takes k-1 columns so the merchant_category_Clothing wasn't included as it can be deduced if a row had 4 0s in all other category columns**

> In order not to confuse:
> - `log_amount`: Compresses wide amount range using `log(amount + 1)` for better model stability
> - `relevant_amount`: Amount normalized within merchant category 

**Regarding Y_train , Y_val and Y_test, they only contain is_fraud which is the target variable**



### Feature Scaling

Since some of the classification algorithms used in this project are sensitive to feature magnitude, feature scaling is applied to ensure that all numerical variables contribute equally during model training. Without scaling, features with larger ranges could dominate the learning process and bias the model.

In this work, scaling is applied only to continuous numerical features such as transaction amount–related variables, velocity measures, age, and trust score. Binary and one-hot encoded features are not scaled, as they are already in a standardized 0–1 format.

Standardization (z-score normalization) is used, where each feature is transformed to have a mean of 0 and a standard deviation of 1. This method is suitable for models like Logistic Regression, which rely on gradient-based optimization. Tree-based models such as Random Forest and XGBoost are trained on the unscaled data, as they are not affected by feature scale.

To avoid data leakage, the scaler is fitted only on the training data and then applied to the validation and test sets.

In [7]:
num_cols = [
    'transaction_hour',
    'device_trust_score',
    'velocity_last_24h',
    'cardholder_age',
    'log_amount',
    'velocity_per_hour',
    'relevant_amount'
]

X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

scaler = StandardScaler()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val_scaled[num_cols]   = scaler.transform(X_val[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test[num_cols])

print("Feature scaling done successfully")

Feature scaling done successfully


### Verification and Sanity Checks

In [8]:
print("Shapes:")
print("X_train:", X_train.shape)
print("X_train_scaled:", X_train_scaled.shape)

print("\nMissing values:")
print(X_train_scaled.isnull().sum().sum())

print("\nColumn consistency:")
print(set(X_train.columns) == set(X_val.columns) == set(X_test.columns))

print("\nScaled feature stats (train):")
print(X_train_scaled[num_cols].describe().loc[['mean', 'std']])

Shapes:
X_train: (6999, 16)
X_train_scaled: (6999, 16)

Missing values:
0

Column consistency:
True

Scaled feature stats (train):
      transaction_hour  device_trust_score  velocity_last_24h  cardholder_age  \
mean      6.700360e-17       -1.258856e-16      -4.974510e-17   -1.949196e-16   
std       1.000071e+00        1.000071e+00       1.000071e+00    1.000071e+00   

        log_amount  velocity_per_hour  relevant_amount  
mean -7.004922e-17       3.350180e-17         0.000000  
std   1.000071e+00       1.000071e+00         1.000071  


### Export Processed Data and Artifacts

In [9]:
X_train_scaled.to_csv("../data/interim/scaled/X_train.csv", index=False)
X_val_scaled.to_csv("../data/interim/scaled/X_val.csv", index=False)
X_test_scaled.to_csv("../data/interim/scaled/X_test.csv", index=False)

X_train.to_csv("../data/interim/unscaled/X_train.csv", index=False)
X_val.to_csv("../data/interim/unscaled/X_val.csv", index=False)
X_test.to_csv("../data/interim/unscaled/X_test.csv", index=False)

Y_train.to_csv("../data/interim/label/Y_train.csv", index=False)
Y_val.to_csv("../data/interim/label/Y_val.csv", index=False)
Y_test.to_csv("../data/interim/label/Y_test.csv", index=False)

joblib.dump(scaler, "scaler.pkl")

print("Data and scaler saved successfully.")

Data and scaler saved successfully.
